# Demo — CLIP-DISSECT

### Interpreting hidden neurons with CLIP and natural language concepts

This notebook is a demo of the **CLIP-DISSECT** pipeline from the ICLR 2023 Paper: [*Automatic Description of Neuron Representations in Deep Vision Networks*](https://arxiv.org/pdf/2204.10965) and the [official codebase](https://github.com/Trustworthy-ML-Lab/CLIP-dissect).

* In the original paper, the authors uses large probe datasets such as Broden/ImageNet and evaluates many model layers.

* Here we use a small probe split and a compact concept set so the full idea is visible in one Colab session.

* This demo is designed for 2026 Schmidt Sciences AIMS Workshop Tutorial on *Towards Trustworthy AI: A Principled Interpretability Perspective*.

*Acknowledgement*: The content is adapted from the materials in SP25 and SP26 UC San Diego's DSC 291 [Trustworthy Machine Learning](https://lilywenglab.github.io/DSC-291-sp26/) course, taught by Prof. Lily Weng and TAs Jerry Thomas John, Divyansh Srivastava, and Atharv Nair.  


## Setup

On Colab, set `Runtime > Change runtime type > GPU` for faster execution. The notebook can also run on CPU with smaller `PROBE_SIZE` values.

In [ ]:
# Install the CLIP package and the lightweight libraries needed for this demo.
!pip -q install git+https://github.com/openai/CLIP.git torchvision tqdm pandas matplotlib

In [ ]:
import math
from pathlib import Path

import clip
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

# Use GPU when available; all model forward passes below use this device.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("torch:", torch.__version__)

## Configuration

If GPU is not available, start small. The `PROBE_SIZE` is a hyper-parameter that can be played with after understanding the pipeline.

In [ ]:
# Keep the probe set small for a live demo; larger values give more stable neuron descriptions.
PROBE_SIZE = 1000 if DEVICE.type == "cuda" else 32
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8

# Target model/layer: these are the neurons we will describe using CLIP.
TARGET_MODEL_NAME = "resnet50"
TARGET_LAYER = "layer4"

# CLIP provides the shared image-text embedding space used for concept matching.
CLIP_MODEL_NAME = "ViT-B/32"

# Pool spatial CNN activations so each channel becomes one scalar activation per image.
POOL_MODE = "avg"
TOP_K = 5

DATA_DIR = Path("clip_dissect_demo_data")
OUTPUT_DIR = Path("clip_dissect_demo_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print({
    "probe_dataset": "CIFAR-100 validation/test split",
    "probe_size": PROBE_SIZE,
    "batch_size": BATCH_SIZE,
    "target_model": TARGET_MODEL_NAME,
    "target_layer": TARGET_LAYER,
    "clip_model": CLIP_MODEL_NAME,
    "pool_mode": POOL_MODE,
})

## Step 1 — Define a Concept Set
Concept set is an open-ended set specified by users and stakeholders. It can be easily expanded and updated in CLIP-Dissect.

* Here, we use 40 candidate concept to demonstrate the pipeline. In the paper, the authors use much more concepts (e.g. 10k-20k English words)

In [ ]:
# Candidate text labels that CLIP-DISSECT can assign to target-model units.
# The best concept for a unit is chosen by comparing these text concepts against images that activate the unit.
concepts = [
    "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck",
    "animal", "vehicle", "wheels", "fur", "feathers", "water", "sky", "grass", "road", "building",
    "person", "tree", "flower", "face", "texture", "striped", "round", "red", "blue", "green",
    "metal", "wood", "window", "tail", "head", "leg", "eye", "smooth", "dark", "bright",
]

print(f"{len(concepts)} concepts")

## Step 2 — Load CIFAR-100 Probing Images

The probing dataset is not the training dataset for CLIP-DISSECT (in fact, CLIP-Dissect is a training-free method). Instead, it is a set of images used to probe the target model internals in order to understand the functionalties of the units.

* The probing dataset is used to answer: *which images activate this neuron, and which text concepts does CLIP associate with those same images?*

* Here, we use CIFAR-100 because it is the smallest built-in probe option from the original script and is practical for a live Colab demo.

In [ ]:
def load_probe_images(probe_size=64):
    # The probe set is only used to observe activations, not to train the model
    raw_dataset = datasets.CIFAR100(root=str(DATA_DIR), train=False, download=True)
    subset_size = min(probe_size, len(raw_dataset))
    indices = list(range(subset_size))
    return Subset(raw_dataset, indices), raw_dataset.classes, indices

probe_dataset, cifar100_classes, probe_indices = load_probe_images(PROBE_SIZE)
print("probe dataset: CIFAR-100")
print("probe images used:", len(probe_dataset))
print("total CIFAR-100 test images:", 10_000)

In [ ]:
def label_name(label):
    return cifar100_classes[int(label)] if cifar100_classes is not None else str(label)

def show_probe_examples(dataset, n=8):
    # Quick sanity check: verify that the probe images and class labels loaded correctly.
    n = min(n, len(dataset))
    fig, axes = plt.subplots(1, n, figsize=(1.8 * n, 2.2))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, range(n)):
        image, label = dataset[idx]
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(label_name(label), fontsize=18)
    plt.tight_layout()

show_probe_examples(probe_dataset, n=8)

## Step 3 — Load CLIP and the Target Vision Model

The target model is the model we want to interpret. CLIP supplies image and text embeddings used to name the target model's neurons.

The original CLIP-DISSECT code uses OpenAI CLIP's `clip.load(...)`, then calls `encode_image(...)` and `encode_text(...)`. We follow that pattern here.

In [ ]:
def load_target_model(name, device):
    if name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
    elif name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        model = models.resnet50(weights=weights)
    else:
        raise ValueError(f"Unsupported target model: {name}")
    model = model.to(device).eval()
    return model, weights.transforms()

target_model, target_preprocess = load_target_model(TARGET_MODEL_NAME, DEVICE)

clip_model, clip_preprocess = clip.load(CLIP_MODEL_NAME, device=DEVICE)
clip_model.eval()
print("models loaded")

In [ ]:
# view the architecture of the target_model
print(target_model)

In [ ]:
# Loop through all modules and print their names
for name, layer in target_model.named_modules():
    # We filter out empty names (the empty string '' represents the whole model)
    if name:
        print(f"Layer Name: {name}  |  Type: {type(layer).__name__}")

## Step 4 — Collect Target Activations and CLIP Image Features

This mirrors the original repository's activation collection:

1. build two versions of the same probe dataset, one with CLIP preprocessing and one with target-model preprocessing,
2. register a forward hook on the target layer,
3. run the target model once per batch,
4. average CNN activations over spatial positions so each channel becomes one unit,
5. encode the same probe images with CLIP using `clip_model.encode_image(...)`.

In [ ]:
def make_transformed_subset(indices, preprocess):
    # Return a subset of dataset
    dataset = datasets.CIFAR100(root=str(DATA_DIR), train=False, download=True, transform=preprocess)
    return Subset(dataset, indices)

probe_clip_dataset = make_transformed_subset(probe_indices, clip_preprocess)
probe_target_dataset = make_transformed_subset(probe_indices, target_preprocess)

def get_activation(outputs, mode):
    """
    Builds a forward hook that extracts one activation vector per input image.

    For CNN layers, the raw output is usually B x C x H x W. We reduce the spatial
    dimensions using either average pooling or max pooling, giving one C-dimensional
    vector per image.

    For ViT-style outputs, the activation may be B x tokens x dim, so we use the
    CLS token as the image-level representation. If the output is already B x dim,
    it is stored directly.
    """
    if mode == "avg":
        def hook(_model, _inputs, output):
            if len(output.shape) == 4:       # CNN feature map: B x C x H x W
                outputs.append(output.mean(dim=[2, 3]).detach())
            elif len(output.shape) == 3:     # ViT tokens: B x tokens x dim
                outputs.append(output[:, 0].clone())
            elif len(output.shape) == 2:     # Already pooled: B x dim
                outputs.append(output.detach())
    elif mode == "max":
        def hook(_model, _inputs, output):
            if len(output.shape) == 4:
                outputs.append(output.amax(dim=[2, 3]).detach())
            elif len(output.shape) == 3:
                outputs.append(output[:, 0].clone())
            elif len(output.shape) == 2:
                outputs.append(output.detach())
    else:
        raise ValueError("mode must be 'avg' or 'max'")
    return hook

def get_module(model, dotted_name):
    # Resolve names like "layer4" or "layer4.2.conv3" into actual PyTorch modules.
    module = model
    for part in dotted_name.split("."):
        module = getattr(module, part)
    return module

@torch.no_grad()
def collect_features(clip_dataset, target_dataset, batch_size=BATCH_SIZE):
    """
    Collects two aligned feature matrices over the same probe images.

    The CLIP dataset is passed through the CLIP image encoder to produce image
    embeddings. The target dataset is passed through the target model, while a
    forward hook records activations from the chosen internal layer.

    Because both datasets use the same image indices and the DataLoaders are not
    shuffled, row i in `image_features` and row i in `target_features` correspond
    to the same original CIFAR-100 image.

    The hook is removed in the `finally` block to prevent duplicate activations
    if this function is run multiple times in the notebook.
    """
    clip_loader = DataLoader(clip_dataset, batch_size=batch_size, shuffle=False)
    target_loader = DataLoader(target_dataset, batch_size=batch_size, shuffle=False)

    image_features = []
    target_features = []

    # The hook captures activations from TARGET_LAYER during the target-model forward pass.
    module_name = get_module(target_model, TARGET_LAYER)
    print("Target_layer : ",TARGET_LAYER)
    # print(module_name)

    hook_handle = module_name.register_forward_hook(get_activation(target_features, POOL_MODE))
    try:
        for (clip_images, _), (target_images, _) in tqdm(zip(clip_loader, target_loader), total=len(target_loader), desc="probe images"):
            # CLIP image embedding: one vector per probe image.
            image_features.append(clip_model.encode_image(clip_images.to(DEVICE)).detach().cpu())

            # Target forward pass triggers the hook above, which appends target activations.
            _ = target_model(target_images.to(DEVICE))
    finally:
        # Always remove hooks; otherwise repeated runs would duplicate captured activations.
        hook_handle.remove()

    return torch.cat(image_features), torch.cat([feat.cpu() for feat in target_features])

image_features, target_features = collect_features(probe_clip_dataset, probe_target_dataset)
print("CLIP image features:", tuple(image_features.shape))
print("target activations:", tuple(target_features.shape))

## Step 5 — Embed Text Concepts with CLIP

The original repository tokenizes raw concept strings with `clip.tokenize(...)` and embeds them with `clip_model.encode_text(...)`. We use the same pattern here.

In [ ]:
@torch.no_grad()
def embed_concepts(concepts, batch_size=BATCH_SIZE):
    # Convert text concepts into CLIP text embeddings.
    text = clip.tokenize(concepts).to(DEVICE)
    features = []
    for start in tqdm(range(0, len(text), batch_size), desc="concept text"):
        features.append(clip_model.encode_text(text[start:start + batch_size]).detach().cpu())
    return torch.cat(features)

text_features = embed_concepts(concepts)
print("CLIP text features:", tuple(text_features.shape))

## Step 6 — Neuron Identification: Scoring Concepts for Each Unit

This is the key step in CLIP-Dissect to identify the best concepts describe a neuron's functionality using different similarity functions.

* In the paper, the authors investigate multiple different similarity functions
* Here, we look at 3 similarity functions: `cos_similarity`, `wpmi` and `soft_wpmi`
* A neuron's explanation is identified as the *concept* (which is a text description) that result in largest similarity value

In [ ]:
# 1. Define concept activation matrix using CLIP (i.e CLIP matrix)
def build_clip_feats(image_features, text_features):
    # CLIP image-text cosine similarities: rows are images, columns are concepts.
    image_features = image_features.float()
    text_features = text_features.float()
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return image_features @ text_features.T

def clear_device_cache():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

#2.1 Define Similarity function 1: cosine
def cos_similarity(clip_feats, target_feats, device='cuda'):
    # Baseline score: correlate each target unit with each CLIP concept over probe images.
    with torch.no_grad():
        clip_feats = clip_feats / torch.norm(clip_feats, p=2, dim=0, keepdim=True)
        target_feats = target_feats / torch.norm(target_feats, p=2, dim=0, keepdim=True)

        batch_size = 10000

        similarities = []
        for t_i in tqdm(range(math.ceil(target_feats.shape[1]/batch_size))):
            curr_similarities = []
            curr_target = target_feats[:, t_i*batch_size:(t_i+1)*batch_size].to(device).T

            for c_i in range(math.ceil(clip_feats.shape[1]/batch_size)):
              curr_similarities.append(curr_target @ clip_feats[:, c_i*batch_size:(c_i+1)*batch_size].to(device))
            similarities.append(torch.cat(curr_similarities, dim=1))
    return torch.cat(similarities, dim=0)

#2.2 Define Similarity function 2: Weighted point-wise mutual information (WPMI)
def wpmi(clip_feats, target_feats, top_k=28, a=2, lam=0.6, device='cuda', min_prob=1e-7):
  # Simpler WPMI variant: uses hard top-k images rather than the soft weighting above.
  with torch.no_grad():
      torch.cuda.empty_cache()
      clip_feats = torch.nn.functional.softmax(a*clip_feats, dim=1)
      # For each neuron, get the indices of the top-k images that activate it most
      inds = torch.topk(target_feats, dim=0, k=top_k)[1]
      # Store concept scores for each neuron.

      prob_d_given_e = []

      for orig_id in tqdm(range(target_feats.shape[1])):
          torch.cuda.empty_cache()
          #Get CLIP concept probabilities for this neuron's top-k activating images.
          curr_clip_feats = clip_feats.gather(0, inds[:,orig_id:orig_id+1].expand(-1,clip_feats.shape[1])).to(device)
          # Sum log concept probabilities across the top-k images.
          curr_p_d_given_e = torch.sum(torch.log(curr_clip_feats+min_prob), dim=0, keepdim=True)
          prob_d_given_e.append(curr_p_d_given_e)

      # first term : Stack all neuron-concept scores into a units x concepts matrix
      prob_d_given_e = torch.cat(prob_d_given_e, dim=0)
      #second term : Estimate common concepts that occur across all neurons. We want to penalize this
      prob_d = (torch.logsumexp(prob_d_given_e, dim=0, keepdim=True) - torch.log(prob_d_given_e.shape[0]*torch.ones([1]).to(device)))

      mutual_info = prob_d_given_e - lam*prob_d
  return mutual_info


#2.3 Define Similarity function 3: Soft Weighted point-wise mutual information (Soft-WPMI)
def soft_wpmi(clip_feats, target_feats, top_k=100, a=10, lam=1, device='cuda',
                        min_prob=1e-7, p_start=0.998, p_end=0.97):
    # Main CLIP-DISSECT scoring rule using soft WMPI.
    with torch.no_grad():
        torch.cuda.empty_cache()

        # Convert image-concept similarities into a probability-like distribution over concepts.
        clip_feats = torch.nn.functional.softmax(a*clip_feats, dim=1)

        # For each unit, find the probe images with the largest target-model activation.
        inds = torch.topk(target_feats, dim=0, k=top_k)[1]
        prob_d_given_e = []

        # Soft weighting prevents the top-k images from being treated as equally informative.
        p_in_examples = p_start-(torch.arange(start=0, end=top_k)/top_k*(p_start-p_end)).unsqueeze(1).to(device)
        for orig_id in tqdm(range(target_feats.shape[1])):

            curr_clip_feats = clip_feats.gather(0, inds[:,orig_id:orig_id+1].expand(-1,clip_feats.shape[1])).to(device)

            curr_p_d_given_e = 1+p_in_examples*(curr_clip_feats-1)
            curr_p_d_given_e = torch.sum(torch.log(curr_p_d_given_e+min_prob), dim=0, keepdim=True)
            prob_d_given_e.append(curr_p_d_given_e)
            torch.cuda.empty_cache()

        prob_d_given_e = torch.cat(prob_d_given_e, dim=0)

        # Background correction: penalize concepts that score highly for many units.
        prob_d = (torch.logsumexp(prob_d_given_e, dim=0, keepdim=True) -
                  torch.log(prob_d_given_e.shape[0]*torch.ones([1]).to(device)))
        mutual_info = prob_d_given_e - lam*prob_d
    return mutual_info

# 3. Get the Concept activation matrix (this step), and the target features (from step 4) and ready to calculate the similarity scores for a neuron
clip_feats = build_clip_feats(image_features, text_features)
target_feats = target_features.float()
print("clip_feats:", tuple(clip_feats.shape), "= images x concepts")
print("target_feats:", tuple(target_feats.shape), "= images x units")

In [ ]:
# 4. Define utility function
def summarize(scores, concepts, top_k=5, method=None):
    # Convert the score matrix into a readable table: one row per target unit.
    scores = scores.detach().cpu()
    top_values, top_ids = scores.topk(k=min(top_k, scores.shape[1]), dim=1)
    rows = []
    for unit in range(scores.shape[0]):
        rows.append({
            "method": method,
            "unit": unit,
            "best_concept": concepts[int(top_ids[unit, 0])],
            "best_score": float(top_values[unit, 0]),
            "top_concepts": ", ".join(concepts[int(idx)] for idx in top_ids[unit]),
        })
    return pd.DataFrame(rows).sort_values("best_score", ascending=False)

similarity_functions = {
    "soft_wpmi": soft_wpmi,
    "wpmi": wpmi,
    "cos_similarity": cos_similarity,

}


# 5. Start calculating scores (identifying neuron concepts) using different similartiy function for all the neuron in this TARGET_LAYER
all_results = {}
for name, fn in similarity_functions.items():
    print(f"\nRunning {name}")
    # Run each scoring function on the same CLIP-concept and target-activation matrices.
    scores_for_method = fn(clip_feats.clone(), target_feats.clone(), device=DEVICE)
    all_results[name] = summarize(scores_for_method, concepts, TOP_K, method=name)

# 6. Use soft-WPMI for the rest of the demo because it is the original CLIP-DISSECT default.
results = all_results["soft_wpmi"]
# results = all_results["wpmi"]
# results = all_results["cos_similarity"]


In [ ]:
# 7. result of soft_wpmi
all_results["soft_wpmi"]

In [ ]:
# 8. result of wpmi
all_results["wpmi"]

In [ ]:
# 9. result of cos similarity
all_results["cos_similarity"]

In [ ]:
# 10. Compare the top-described units under the three scoring rules.
comparison = pd.concat([df.head(5) for df in all_results.values()], ignore_index=True)
comparison[["method", "unit", "best_concept"]]

In [ ]:
# 11. Save the neuron-description tables so they can be inspected outside the notebook.
for method, df in all_results.items():
    csv_path = OUTPUT_DIR / f"clip_dissect_descriptions_{TARGET_LAYER}_{method}.csv"
    df.to_csv(csv_path, index=False)
    print("saved:", csv_path)

## Step 7 — Inspect One Unit Qualitatively

The result tables in Step 6 gives a text description.
* By default, the notebook uses `soft_wpmi` for qualitative inspection because the authors found it performs the best (thus it is set as the original script's default similarity function).
* Below, we visualize whether the top-activating images look compatible with that description.
* We also visualize what are the most frequenct concepts appearing in the neurons of the TARGET_LAYER in the model


### Visualization 1: Top Activation images for the most interpretable neurons

In [ ]:
N_UNITS = 10
TOP_IMAGES = 8

def show_unit_grid(results, dataset, target_features, n_units=10, n_images=8, unit_ids=None):
      if unit_ids is None:
          selected = results.head(n_units).copy()
      else:
          selected = results[results["unit"].isin(unit_ids)].copy()
          selected["unit"] = pd.Categorical(selected["unit"], categories=unit_ids, ordered=True)
          selected = selected.sort_values("unit")

      fig, axes = plt.subplots(
          len(selected),
          n_images,
          figsize=(1.7 * n_images, 1.85 * len(selected)),
          squeeze=False,
      )

      for row_idx, (_, row) in enumerate(selected.iterrows()):
          unit = int(row["unit"])

          # Images with the largest activations are the qualitative evidence for the unit description.
          values, ids = target_features[:, unit].topk(k=min(n_images, len(dataset)))

          for col_idx in range(n_images):
              ax = axes[row_idx, col_idx]
              ax.axis("off")

              if col_idx < len(ids):
                  image, _ = dataset[int(ids[col_idx])]
                  ax.imshow(image)

              # Optional annotation block if you want to label each row directly on the grid.
              if col_idx == 0:
                  ax.text(
                      0.02,
                      0.08,
                      f"Unit {unit}\n{row['best_concept']}",
                      transform=ax.transAxes,
                      fontsize=10,
                      fontweight="bold",
                      color="white",
                      bbox=dict(facecolor="black", alpha=0.65, edgecolor="none", pad=3),
                  )

      plt.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.08, hspace=0.18)
      plt.show()

show_unit_grid(results=results,dataset=probe_dataset,target_features=target_features,n_units=N_UNITS,n_images=TOP_IMAGES)
# show_unit_grid(results=results,dataset=probe_dataset,target_features=target_features,n_units=N_UNITS,n_images=TOP_IMAGES,unit_ids=[10,232,343,42,50])

### Visualization 2: Most frequent concepts appears in the neurons of the TARGET_LAYER in the model

In [ ]:
# get the number of unit in TARGET_LAYER
num_unit = target_feats.shape[1]
print(f"TOTAL NUMBER of neuron unit in {TARGET_LAYER} is {target_feats.shape[1]}")

In [ ]:
HISTOGRAM_TOP_N = 20
counts = results["best_concept"].value_counts().head(HISTOGRAM_TOP_N)

# 1. Create the primary plot setup (Left Y-axis)
fig, ax1 = plt.subplots(figsize=(12, 5))

# Plot the bars on the primary axis
bars = ax1.bar(
    counts.index,
    counts.values,
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.5,
)

# Configure the Left Y-axis (Number of Units)
ax1.set_xlabel("best concept", fontweight="bold", labelpad=10)
ax1.set_ylabel("number of units", color="#4C78A8", fontweight="bold")
ax1.tick_params(axis="y", labelcolor="#4C78A8")
ax1.set_xticklabels(counts.index, rotation=45, ha="right")
ax1.grid(axis="y", alpha=0.25)

# 2. Create the twin plot setup (Right Y-axis: % of Units in the TARGET_LAYER)
ax2 = ax1.twinx()

# Calculate the limits for the right axis based on the left axis values
# This ensures the grids line up perfectly without drawing a second overlapping bar layer
min_val, max_val = ax1.get_ylim()
ax2.set_ylim((min_val / num_unit * 100), (max_val / num_unit * 100))

# Configure the Right Y-axis (% of units)
ax2.set_ylabel("% of units in the layer", color="#2c3e50", fontweight="bold")
ax2.tick_params(axis="y", labelcolor="#2c3e50")

# 3. Final touches
plt.title(f"Most common CLIP-DISSECT concepts in {TARGET_LAYER}", fontsize=18, pad=15)
plt.tight_layout()
plt.show()

## Step 8:

### To examine different layers
* change the config
* only need to re-run Step 4 (collect target acitvation), Step 6 (score concepts for each unit), Step 7 (qualitative result)

### Save results

* results in Step 6 are saved to `OUTPUT_DIR` (see the left bar "folder" icon)
* note: the folder is the local folder and will be cleared out when runtime is disconnected



## Future Reading and Demo

If you are interested in learning more about this line of work (*Interpretable Deep Learning, Mechanistic Interpretability, Automated Interpretability*), below are a few pointers and papers for future readings.  

**(i) Neuron Interpretability Tools for Vision, Language, and Audio Models:**

* [1] Oikarinen and Weng, CLIP-Dissect: Automatic Description of Neuron Representations in Deep Vision Networks, ICLR 23 (spotlight)

* [2] Oikarinen and Weng, [Linear Explanations for Individual Neurons](https://lilywenglab.github.io/Linear-Explanation/), ICML 24

* [3] Bai & Iyer etal, [Describe-and-Dissect: Interpreting Neurons in Vision Networks with Language Models](https://lilywenglab.github.io/Describe-and-Dissect/index.html), TMLR 25

* [4] Wu & Lin etal, [AND: Audio Network Dissection for Interpreting Deep Acoustic Models](https://lilywenglab.github.io/Audio_Network_Dissection/), ICML 24


**(ii) Learning Inherently Interpretable neural network models:**

* [5] Oikarinen etal, [Label-Free Concept Bottleneck Models](https://openreview.net/pdf?id=FlCg47MNvBA), ICLR 23

* [6] Srivastava & Yan etal, [VLG-CBM: Training Concept Bottleneck Models with Vision-Language Guidance](https://lilywenglab.github.io/VLG-CBM/), NeurIPS 24

* [7] Sun etal, [Concept Bottleneck Large Language Models](https://lilywenglab.github.io/CB-LLMs/), ICLR 25

* [8] Kulkarni etal, [Interpretable Generative Models through Post-hoc Concept Bottlenecks](https://lilywenglab.github.io/posthoc-generative-cbm/), CVPR 25

* [9] Kulkarni etal, [Interpretable and Steerable Concept Bottleneck Sparse Autoencoders](https://arxiv.org/pdf/2512.10805), CVPR 26

**(iii) Evaluating Faithfulness of Neuron Explanations:**

* [10] Oikarinen etal, [Evaluating Neuron Explanations: A unified Framework with Sanity Checks](https://lilywenglab.github.io/Neuron_Eval/), ICML 25  
